In [ ]:
# Install OpenMPI and mpi4py
!apt-get update -y
!apt-get install -y libopenmpi-dev openmpi-bin
!pip install mpi4py


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,642 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,712 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-s

In [16]:
%%writefile parallel_quicksort.py
from mpi4py import MPI
import numpy as np
import time

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

def quicksort(arr):
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return quicksort(left) + middle + quicksort(right)

# Start timer
start_time = time.time()

# Master process manually defines data
if rank == 0:
    # ✅ Manually set your array here
    data = np.array([10, 3, 5, 2, 8, 6, 1, 9, 4, 7, 15, 12, 14, 11, 13, 19, 18, 17, 16, 20])
    chunks = np.array_split(data, size)
else:
    chunks = None

# Distribute data to all processes
chunk = comm.scatter(chunks, root=0)

# Local quicksort
sorted_chunk = quicksort(chunk)

# Gather sorted chunks
gathered = comm.gather(sorted_chunk, root=0)

if rank == 0:
    from heapq import merge
    final_sorted = list(gathered[0])
    for part in gathered[1:]:
        final_sorted = list(merge(final_sorted, part))

    end_time = time.time()

    # Print first 10 elements as normal integers (removing np.int64)
    print("Sorted first 20 elements:", [int(x) for x in final_sorted[:20]])
    print(f"Total time with {size} processes: {end_time - start_time:.4f} seconds")


Overwriting parallel_quicksort.py


In [17]:
!OMPI_ALLOW_RUN_AS_ROOT=1 OMPI_ALLOW_RUN_AS_ROOT_CONFIRM=1 \
 mpiexec --allow-run-as-root --oversubscribe -n 4 python3 parallel_quicksort.py

Sorted first 20 elements: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
Total time with 4 processes: 0.0866 seconds
